In [1]:
import sys
import os
import random
import torch

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

from cvae_generator import CVAEGenerator

SEQ_LENGTH = 6
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

# --- Generate temporary peptide data ---
random.seed(42)
peptides = list(set(
    ''.join(random.choices(AMINO_ACIDS, k=SEQ_LENGTH))
    for _ in range(500)
))
print(f"Generated {len(peptides)} unique temporary peptides")
print(f"Example sequences: {peptides[:5]}")

# --- One-hot encode ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

one_hot = encode(peptides, SEQ_LENGTH)
print(f"Tensor shape: {one_hot.shape}")  # Expected: [n, 120]

# --- Initialize model ---
cvae = CVAEGenerator(input_dim=INPUT_DIM, latent_dim=16, hidden_dim=64,condition_dim= 32)
print(f"Device:     {cvae.device}")
print(f"Input dim:  {cvae.input_dim}")
print(f"Latent dim: {cvae.latent_dim}")
print(f"Hidden dim: {cvae.hidden_dim}")
print(f"Condition dim: {cvae.condition_dim}")
constraints = {
    "length": {"min": 8, "max": 14},
    "net_charge_pH5_5": {"min": 1.0, "max": 6.0},
    "hydrophobicity": {"min": -1.0, "max": 0.5},
}

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints,
    count=one_hot.shape[0],
    device=cvae.device,
)
# --- Train ---
one_hot = one_hot.to(cvae.device)
cvae.train_model(one_hot, epochs=300, batch_size=64, lr=1e-3,conditions=conditions)
print("Training complete!")

# --- Generate ---
generated = cvae.generate_peptides(count=10,constraints = constraints,temperature=0.8)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
    status = "✓" if valid else "✗ INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")

# --- Novelty + diversity check ---
training_set = set(peptides)
novel = [p for p in generated if p not in training_set]
print(f"\nNovel peptides  (not in training data): {len(novel)}/{len(generated)}")
print(f"Unique peptides (diversity check):      {len(unique_generated)}/{len(generated)}")


Generated 500 unique temporary peptides
Example sequences: ['KWNQVS', 'YHCCTP', 'CIADTY', 'GWTNLH', 'WVGPPE']
Tensor shape: torch.Size([500, 120])
Device:     cuda
Input dim:  120
Latent dim: 16
Hidden dim: 64
Condition dim: 32
  Epoch 050/300 | Recon: 1.9989 | KL: 0.5699 | KL weight: 0.49
  Epoch 100/300 | Recon: 1.7569 | KL: 0.4999 | KL weight: 0.99
  Epoch 150/300 | Recon: 1.5783 | KL: 0.5417 | KL weight: 1.00
  Epoch 200/300 | Recon: 1.4544 | KL: 0.5699 | KL weight: 1.00
  Epoch 250/300 | Recon: 1.4057 | KL: 0.5825 | KL weight: 1.00
  Epoch 300/300 | Recon: 1.4098 | KL: 0.5849 | KL weight: 1.00
Training complete!
Peptide 01: LDWCQH  ✓
Peptide 02: EFNTIR  ✓
Peptide 03: SCKDEN  ✓
Peptide 04: IQYAWV  ✓
Peptide 05: LYLCQM  ✓
Peptide 06: QRQKQW  ✓
Peptide 07: KLLRVN  ✓
Peptide 08: IRWYAE  ✓
Peptide 09: DYTPYY  ✓
Peptide 10: HMGQIG  ✓

Novel peptides  (not in training data): 10/10
Unique peptides (diversity check):      10/10
